<a href="https://colab.research.google.com/github/angieapol33-bot/FUNDAI-Laboratories-APOLINAR/blob/main/Lab3_Game_AI_APOLINAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Game AI Using Minimax with Alpha-Beta Pruning

## Fundamentals of Artificial Intelligence

**Name:** Angeline Grace Joy L. Apolinar  
**Course:** BSIT
**Section:** 09282-FUNDAI  
**Date:** September 2, 2026  
**Selected Game:** Tic-Tac-Toe  
**GitHub URL:** https://github.com/angieapol33-bot/FUNDAI-Laboratories-APOLINAR.git

## Description

This laboratory implements a Tic-Tac-Toe AI using the Minimax algorithm with Alpha-Beta pruning.

The game is playable inside Google Colab using `ipywidgets`.

In [1]:
import math
import ipywidgets as widgets
from IPython.display import display

In [2]:
class TicTacToeGame:
    X = "X"
    O = "O"
    EMPTY = " "

    def __init__(self):
        self.board = [self.EMPTY] * 9
        self.current_player = self.X

    def available_moves(self):
        return [i for i, value in enumerate(self.board)
                if value == self.EMPTY]

    def make_move(self, move):
        self.board[move] = self.current_player
        self.current_player = self.O if self.current_player == self.X else self.X

    def undo_move(self, move):
        self.board[move] = self.EMPTY
        self.current_player = self.O if self.current_player == self.X else self.X


    def get_winner(self):
        winning_lines = [
            (0, 1, 2),
            (3, 4, 5),
            (6, 7, 8),
            (0, 3, 6),
            (1, 4, 7),
            (2, 5, 8),
            (0, 4, 8),
            (2, 4, 6)
        ]

        for a, b, c in winning_lines:
            if self.board[a] != self.EMPTY and self.board[a] == self.board[b] == self.board[b] == self.board[c]:
                return self.board[a]

        return None

    def is_draw(self):
        return self.get_winner() is None and len(self.available_moves()) == 0

    def is_terminal(self):
        return self.get_winner() is not None or len(self.available_moves()) == 0

    def utility(self):
        winner = self.get_winner()

        if winner == self.X:
            return 1
        elif winner == self.O:
            return -1
        else:
            return 0

In [3]:
def minimax_alpha_beta(game, alpha=-math.inf, beta=math.inf):
    if game.is_terminal():
        return game.utility(), None

    # MAX player: X
    if game.current_player == TicTacToeGame.X:
        best_value = -math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value > best_value:
                best_value = value
                best_move = move

            alpha = max(alpha, best_value)

            if alpha >= beta:
                break

        return best_value, best_move

    # MIN player: O
    else:
        best_value = math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value < best_value:
                best_value = value
                best_move = move

            beta = min(beta, best_value)

            if alpha >= beta:
                break

        return best_value, best_move

In [4]:
class TicTacToeUI:

    def __init__(self):
        self.game = TicTacToeGame()

        self.buttons = [
            widgets.Button(
                description=" ",
                layout=widgets.Layout(width="60px", height="60px")
            )
            for i in range(9)
        ]

        for i in range(9):
            self.buttons[i].on_click(lambda btn, idx=i: self.on_cell_click(idx))

        self.status = widgets.HTML(value="<b>Human (X) moves first.</b>")

        self.reset_button = widgets.Button(description="Reset", button_style="info")
        self.reset_button.on_click(self.on_reset)

        self.grid = widgets.GridBox(
            children=self.buttons,
            layout=widgets.Layout(
                grid_template_columns="repeat(3, 60px)",
                grid_gap="5px"
            )
        )

        self.widget = widgets.VBox([self.status, self.grid,self.reset_button])
        display(self.widget)

        self.refresh()

    def refresh(self):

        for i, button in enumerate(self.buttons):
            button.description = self.game.board[i]
            button.disabled = self.game.is_terminal() or self.game.board[i] != TicTacToeGame.EMPTY


        if self.game.is_terminal():
            winner = self.game.get_winner()

            if winner:
                self.status.value = f"<b>Player {winner} wins!</b>"
            else:
                self.status.value = "<b>It's a draw!</b>"

        else:
            self.status.value = f"<b>Current player: {self.game.current_player}</b>"


    def on_cell_click(self, index):
        if self.game.board[index] != TicTacToeGame.EMPTY:
            return

        if self.game.is_terminal():
            return

        if self.game.current_player != TicTacToeGame.X:
            return

        # Human move
        self.game.make_move(index)


        # AI move if game is not finished
        if not self.game.is_terminal():
            ai_move = self.get_ai_move()
            if ai_move is not None:
                self.game.make_move(ai_move)

        self.refresh()

    def get_ai_move(self):
        _, move = minimax_alpha_beta(self.game)
        return move

    def on_reset(self, button):
      self.game = TicTacToeGame()
      self.refresh()

In [5]:
TicTacToeUI()

##Algorithm Explanation

### Minimax
The AI evaluates possible future game states.
Player X maximizes utility, while Player O minimizes utility.

### Alpha-Beta Pruning
Alpha-Beta pruning removes branches that cannot change the final decision.
This makes the AI faster while preserving the optimal result.

### Utility
- X wins: +1
- Draw: 8
- O wins: -1

**1. Which player does the AI control?**  
The AI plays as O. You (the human) always take X, and the computer decides O’s moves by running Minimax search enhanced with Alpha-Beta pruning.

**2. What utility values were used?**  
Terminal positions are scored +1 if X has won, –1 if O has won, and 0 if the board is full with no winner (a draw). These three simple numbers let the minimax algorithm rank every possible outcome and pick the move that is best for O.

**3. How does Alpha-Beta pruning improve Minimax?**  
Alpha-Beta pruning cuts away large parts of the game tree that cannot influence the final choice. It maintains two bounds: α (the best score the maximizer has found so far) and β (the best score the minimizer has found so far). As soon as α ≥ β the remaining sibling branches are pruned, because they can never change the decision. The algorithm therefore explores fewer nodes yet still returns the identical optimal move that plain minimax would have chosen.

**4. What happens when the human chooses a move that leads to a draw?**  
If your sequence of moves forces the game into a drawn position, the AI continues to reply with the strongest available replies. When the board is completely filled and neither side has three-in-a-row, the program simply announces “It’s a draw!” Perfectly played Tic-Tac-Toe always ends this way; neither player can force a win against an opponent that never errs.

**5. Why is Tic-Tac-Toe suitable for full Minimax search?**  
The entire game tree of Tic-Tac-Toe is tiny—only a few hundred thousand positions at most. Because the tree is both finite and small, the AI can expand every branch all the way to a terminal leaf without needing any heuristic evaluation function. This makes the game an ideal classroom example for illustrating pure minimax search and the practical speed-up provided by Alpha-Beta pruning.

**Reflection**

**Challenges Encountered**  
- Grasping how the Minimax algorithm evaluates every possible future board state and then linking that recursive logic to the live interactive board felt confusing at first, especially when Alpha-Beta pruning started cutting branches mid-search.

**What I Learned**  
- I now understand that Minimax systematically scores every terminal position and works backward to select the strongest move for the AI, while Alpha-Beta pruning dramatically speeds up the process by discarding any branches that can no longer affect the final decision.